# 05 · Enrich (derived layer)

Adds the stripped derived columns on the canonical file, using frozen reference files.

- `AMOUNT_USD` — full award amount in USD (same on every recipient row), via the ECB **daily**
  rate at the grant `START_DATE`, else 1 Jan of `GRANT_YEAR` (matching the 2024 method); no
  resolvable date -> blank (counts only). Legacy IOI rows use the ECB 2010-2020 average. Needs the
  ECB historical CSV in `reference/`. (CLP is not an ECB currency, so CLP rows stay counts-only.)
- `DASH_AMT` — `AMOUNT_USD` for single-recipient awards, `0` for multi-recipient (split unknown).
  Multi-recipient = >1 distinct OI in the award cluster, or a manual override (DataCite). A
  `MULTI_RECIPIENT` flag is written for transparency.
- `FUNDER_ROR` — funder name->ROR crosswalk on a normalised key. Recipient RORs are out of scope
  this year (~1,500 distinct recipients vs ~75 funders); one line re-enables them if ever needed.
- Backfills `GRANT_YEAR` / `GRANT_DURATION` only where blank.
- `SOLUTION_CATEGORY` — each OI's solution category(ies) via an OI->category crosswalk (one OI
  can have several). `AWARD_SOLUTION_CATEGORY` carries the union across all OIs on a multi-recipient
  award (the 2024 rule). (`GRANT_CAT`/award-type is still produced later by the classifier, not here.)
- `grant_solution_categories.csv` — a separate **award x category** "reach" export: the full award
  `AMOUNT_USD` counted once in every category the award touches. Rows **overlap by design and must
  not be summed to a total** — they measure funding *reaching* each category, not a split of it.
  Totals and $/funder stay in `enriched.csv`, counted once per award.

First run writes reference templates if missing — fill, then re-run.

**Aggregation rule (carry into S1):** per-OI amount = sum(`DASH_AMT`); per-OI count = row count;
ecosystem/funder amount = sum(`AMOUNT_USD`) over **unique `award_cluster_id`**.

In [ ]:
import pandas as pd
from pathlib import Path
import re

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## CONFIG

In [ ]:
BASE_DIR=Path('/content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding')
USE_CORRECTED_INPUT = True   # False for the first pass (before Sarah's review)

IN_DEDUP = BASE_DIR/('deduplicated_clean_categorised_corrected.csv'
                     if USE_CORRECTED_INPUT
                     else 'deduplicated_clean_categorised.csv')
FX_FILE=BASE_DIR/'reference'/'eurofxref-hist.csv'             # ECB daily rates; download once (frozen):
                                                            # https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.csv
ROR_FILE=BASE_DIR/'reference'/'ror_crosswalk.csv'
ROR_EXT_FILE=BASE_DIR/'reference'/'ROR_crosswalk_extended.csv'  # extended: ROR → attributes
OICAT_FILE=BASE_DIR/'reference'/'oi_solution_categories.csv'   # oi_normalised, solution_category (one row per pair)
OUT_ENRICH=BASE_DIR/'enriched.csv'
OUT_CAT=BASE_DIR/'grant_solution_categories.csv'   # award x category 'reach' view (overlaps; never sum to a total)

COL_AMOUNT,COL_CURRENCY='AMOUNT','CURRENCY'
COL_START,COL_END,COL_YEAR,COL_DURATION='START_DATE','END_DATE','GRANT_YEAR','GRANT_DURATION'
COL_FUNDER,COL_RECIPIENT,COL_OI='FUNDER','RECIPIENT','OI'

# Conversion: ECB daily rate at START_DATE, else 1 Jan of GRANT_YEAR (2024 method).
# No resolvable date -> AMOUNT_USD blank (counts only). Legacy rows use the 2010-2020 average.
LEGACY_FLAG=None                       # e.g. ('source_dataset','ioi_legacy') if such rows exist
LEGACY_AVG_YEARS=(2010,2020)
FORCE_MULTI_GRANT_IDS=set()

## Helpers, load, templates

In [ ]:
def norm_name(s): return re.sub(r'\s+',' ',re.sub(r'[.,]',' ',str(s or '').lower())).strip()
def norm_oi(v): return re.sub(r'\s+',' ',str(v or '').strip().lower())
def year_of(row):
    d=pd.to_datetime(row.get(COL_START),errors='coerce')
    if pd.notna(d): return int(d.year)
    y=pd.to_numeric(row.get(COL_YEAR),errors='coerce')
    return int(y) if pd.notna(y) else None

df=pd.read_csv(IN_DEDUP,dtype=str); df[COL_AMOUNT]=pd.to_numeric(df[COL_AMOUNT],errors='coerce')
(BASE_DIR/'reference').mkdir(parents=True,exist_ok=True)
if not ROR_FILE.exists():
    names=df.get(COL_FUNDER,pd.Series(dtype=str)).dropna()   # FUNDERS ONLY (recipients out of scope)
    pd.DataFrame({'name_normalised':sorted({norm_name(n) for n in names if str(n).strip()}),'ror_id':''}).to_csv(ROR_FILE,index=False)
    print('Wrote ROR template (FUNDERS only,',df[COL_FUNDER].nunique(),'distinct):',ROR_FILE)
if not FX_FILE.exists():
    print('!! Missing', FX_FILE)
    print('   Download the ECB historical daily rates CSV and save it there (it is the frozen FX source):')
    print('   https://www.ecb.europa.eu/stats/eurofxref/')
if not OICAT_FILE.exists():
    ois=sorted({norm_oi(x) for x in df[COL_OI].dropna() if str(x).strip()})
    pd.DataFrame({'oi_normalised':ois,'solution_category':''}).to_csv(OICAT_FILE,index=False)
    print('Wrote OI-category template (fill from Infra Finder; add a ROW PER CATEGORY for multi-category OIs):',OICAT_FILE)

## Compute USD, ROR, multi-recipient, DASH_AMT, year/duration, categories

In [ ]:
ror=pd.read_csv(ROR_FILE,dtype=str).dropna(subset=['ror_id']); ror_map=dict(zip(ror['name_normalised'],ror['ror_id']))

# ECB daily reference rates: each currency column = units of that currency per 1 EUR ('USD' = USD per EUR)
ecb=pd.read_csv(FX_FILE); ecb['Date']=pd.to_datetime(ecb['Date'],errors='coerce')
ecb=ecb.dropna(subset=['Date']).set_index('Date').sort_index()
ecb=ecb.apply(pd.to_numeric, errors='coerce')
ecb_dates=ecb.index
legacy_avg=ecb.loc[(ecb.index.year>=LEGACY_AVG_YEARS[0])&(ecb.index.year<=LEGACY_AVG_YEARS[1])].mean()

assert 1.0 < ecb['USD'].median() < 1.7, "ECB USD/EUR out of range — FX file looks wrong"

def rates_asof(d):
    d=pd.to_datetime(d,errors='coerce')
    if pd.isna(d): return None
    pos=ecb_dates.searchsorted(d,side='right')-1     # latest trading day <= d
    return ecb.iloc[pos] if pos>=0 else None

def cross_to_usd(amt,cur,rates):
    if rates is None: return None
    usd=rates.get('USD')
    if pd.isna(usd): return None
    if cur=='EUR': return amt*usd
    cpe=rates.get(cur)
    if cpe is None or pd.isna(cpe) or cpe==0: return None    # currency not in ECB (e.g. CLP) -> counts only
    return amt*usd/cpe

def to_usd(row):
    amt=row.get(COL_AMOUNT)
    if pd.isna(amt): return None
    cur=str(row.get(COL_CURRENCY) or '').upper()
    if cur in ('','USD'): return amt if cur=='USD' else None
    if LEGACY_FLAG and str(row.get(LEGACY_FLAG[0]))==LEGACY_FLAG[1]:
        return cross_to_usd(amt,cur,legacy_avg)
    d=row.get(COL_START)
    if pd.isna(pd.to_datetime(d,errors='coerce')):
        y=pd.to_numeric(row.get(COL_YEAR),errors='coerce')
        d=f'{int(y)}-01-01' if pd.notna(y) else None
    return cross_to_usd(amt,cur,rates_asof(d)) if d is not None else None

df['AMOUNT_USD']=df.apply(to_usd,axis=1)
df['FUNDER_ROR']=df[COL_FUNDER].map(lambda n: ror_map.get(norm_name(n),'')) if COL_FUNDER in df else ''   # funders only this year

oi_n=df.groupby('award_cluster_id')[COL_OI].transform('nunique')
df['MULTI_RECIPIENT']=oi_n>1
if FORCE_MULTI_GRANT_IDS:
    df.loc[df['GRANT_ID'].astype(str).str.strip().isin({str(x).strip() for x in FORCE_MULTI_GRANT_IDS}),'MULTI_RECIPIENT']=True
df['DASH_AMT']=df.apply(lambda r: 0.0 if r['MULTI_RECIPIENT'] else r['AMOUNT_USD'], axis=1)

# OI solution categories (one OI -> possibly several), via crosswalk
oicat=pd.read_csv(OICAT_FILE,dtype=str)
oicat['oi_normalised']=oicat['oi_normalised'].map(norm_oi)
cat_map={}
for oi,g in oicat.groupby('oi_normalised'):
    cats=sorted({str(c).strip() for c in g['solution_category'] if str(c).strip() and str(c)!='nan'})
    if cats: cat_map[oi]=cats
oin=df[COL_OI].map(norm_oi)
df['SOLUTION_CATEGORY']=oin.map(lambda o: '; '.join(cat_map.get(o,[])))
# 2024 rule: a multi-recipient award carries the union of all its OIs' categories
acc={}
for cid,sub in pd.DataFrame({'cid':df['award_cluster_id'],'oi':oin}).groupby('cid'):
    s=set()
    for o in sub['oi']: s.update(cat_map.get(o,[]))
    acc[cid]=sorted(s)
df['AWARD_SOLUTION_CATEGORY']=df['award_cluster_id'].map(lambda c: '; '.join(acc.get(c,[])))

if COL_YEAR in df:
    b=df[COL_YEAR].isna()|(df[COL_YEAR].astype(str).str.strip()=='')
    df.loc[b,COL_YEAR]=df[b].apply(lambda r:'' if year_of(r) is None else str(year_of(r)),axis=1)
if COL_DURATION in df:
    def dur(r):
        s=pd.to_datetime(r.get(COL_START),errors='coerce'); e=pd.to_datetime(r.get(COL_END),errors='coerce')
        return max(1,round((e-s).days/365)) if (pd.notna(s) and pd.notna(e)) else 1
    b=df[COL_DURATION].isna()|(df[COL_DURATION].astype(str).str.strip()=='')
    df.loc[b,COL_DURATION]=df[b].apply(lambda r:str(dur(r)),axis=1)

## OI name canonicalisation
Three case-only variants in the OI column (arXiv/arxiv, bioRxiv/biorxiv, medRxiv/medrxiv) — preprint servers harvested inconsistently across sources.
Left unfixed, every OI-level aggregate splits these across two rows, and the split also corrupts MULTI_RECIPIENT / DASH_AMT below (computed by OI within cluster). Recase only — no OI is remapped to a different infrastructure.

In [ ]:
# %%
OI_CANONICAL = {'arxiv': 'arXiv', 'biorxiv': 'bioRxiv', 'medrxiv': 'medRxiv'}

_before = df[COL_OI].nunique()
df[COL_OI] = df[COL_OI].map(lambda x: OI_CANONICAL.get(str(x).strip().lower(), x))
print(f"OI distinct: {_before} -> {df[COL_OI].nunique()}")

# General guard — catches any NEW case collision a future harvest introduces.
_lower = df[COL_OI].astype(str).str.strip().str.lower()
_coll = {lo: sorted(v.unique()) for lo, v in df.groupby(_lower)[COL_OI] if v.nunique() > 1}
assert not _coll, f"Unresolved OI case-variants: {_coll}"

OI distinct: 97 -> 94


## Within-source duplicate detection (reported check)

Award clustering (02_deduplicate) resolves *multi-recipient* awards: one
grant, several OIs. It does NOT resolve *within-source* duplicates, because the upstream fuzzy match only generates candidate pairs ACROSS source types (scrapers x OpenAlex, scrapers x OpenAIRE, OpenAIRE x OpenAlex). Two rows from the same harvester are never compared.

This matters because OpenAlex reproduces NIH's award identifiers verbatim, and NIH embeds the fiscal support year in the identifier (5u24hg006620-11, 5u24hg006620-12, ...). One project, N rows.

This cell PRINTS rather than acts. If a future harvest introduces another funder with year-suffixed identifiers, whoever runs the pipeline will see it here without knowing to look for it.

In [ ]:
NIH_SUPPORT_YEAR = re.compile(r"^\d([a-z]\d{2}[a-z]{2}\d{6})-\d+", re.IGNORECASE)


def nih_core_project(grant_id) -> str | None:
    """NIH grant IDs: {app_type}{activity}{institute}{serial}-{support_year}."""
    m = NIH_SUPPORT_YEAR.match(str(grant_id))
    return m.group(1).lower() if m else None


def report_within_source_lookalikes(df: pd.DataFrame) -> pd.DataFrame:
    """Rows that look like the same award reported more than once by one source.

    Key: source + funder + recipient + title + start date + OI, with two or
    more DISTINCT grant IDs. Same grant ID against different OIs is a
    multi-recipient award and is expected — not flagged.
    """
    key_cols = ["source_dataset", "FUNDER_ROR", "RECIPIENT", "TITLE", "START_DATE", "OI"]
    work = df.copy()
    work["_key"] = (
        work[key_cols].astype(str).apply(lambda r: "||".join(r.str.lower()), axis=1)
    )

    groups = work.groupby("_key").agg(
        n_rows=("record_uid", "size"),
        n_grant_ids=("GRANT_ID", "nunique"),
        grant_ids=("GRANT_ID", lambda s: "; ".join(sorted(map(str, s)))),
        funder=("FUNDER", "first"),
        source=("source_dataset", "first"),
        oi=("OI", "first"),
        usd=("AMOUNT_USD", "sum"),
    )
    flagged = groups[(groups.n_rows > 1) & (groups.n_grant_ids > 1)]

    print(f"Within-source lookalike groups: {len(flagged)}")
    print(f"Excess rows: {int(flagged.n_rows.sum() - len(flagged))}")
    print(f"USD in flagged groups: ${flagged.usd.sum():,.0f}\n")
    if len(flagged):
        with pd.option_context("display.max_colwidth", 60, "display.width", 200):
            print(
                flagged[["source", "funder", "oi", "n_rows", "grant_ids", "usd"]]
                .sort_values("usd", ascending=False)
                .to_string(index=False)
            )
    return flagged.reset_index(drop=True)


lookalikes = report_within_source_lookalikes(df)

# Expected as of the 2026-07 harvest: 7 groups / 12 excess rows.
#   - 4 groups, 9 rows : NIH fiscal support-year increments  -> collapsed below
#   - 1 group,  1 row  : CIHR 189339_1 / 189353_1, identical $25K
#                        companion grants to co-PIs, or a genuine duplicate
#   - 1 group,  1 row  : DFG 53121094 / 52691141, ICPSR summer programme,
#                        identical amount and dates
#   - 1 group,  1 row  : NSF 9318536 / 9318521, "Collaborative Research",
#                        DIFFERENT amounts and end dates -> two real awards
# Decision (ET, 2026-07): collapse NIH only. The other three (~$0.19M) are
# retained under the 2024 rule: near-perfect matches with differing amounts
# are kept, on the assumption that sub-awards are correctly assigned.
if len(lookalikes) > 7:
    print("\n!! More lookalike groups than at handover. Adjudicate before analysis.")

# 13/07/2026: 25 groups flagged - many look like false positives (null-collisions, not duplications); only the 3 NIH ones are actual fiscal increments to collapse in the next cell.

Within-source lookalike groups: 25
Excess rows: 38
USD in flagged groups: $73,111,303

  source                                         funder                                       oi  n_rows                                                                                               grant_ids          usd
openalex                [National Institutes of Health]                                    icpsr       4                                    2p01ag020166-07a2; 5p01ag020166-08; 5p01ag020166-10; 5p01ag020166-11 1.674516e+07
     HHS        Department of Health and Human Services                                    icpsr       3                                                                  NU58DP006956; R21HD104993; U24AG076462 1.144554e+07
scrapers                     Chan Zuckerberg Initiative                                  medRxiv       3    chanzuckerberg::0061K00000f965KQAQ; chanzuckerberg::a0C1K00000Y8rWL; chanzuckerberg::a0C1K00000Ynwi3 7.898101e+06
scrapers                 

## Collapse NIH fiscal-year increments

NIH awards arrive from OpenAlex as one row per fiscal support year, each carrying that year's obligation (verified non-cumulative: Galaxy u24hg006620 runs 1.91 / 1.87 / 1.86 / 1.50 across years -10 to -13).

We collapse to the core project number and SUM the increments. Award COUNT falls; total USD is unchanged. This aligns NIH's grain with the HHS rows from usaspending, which carry cumulative project totals.

KNOWN LIMIT — carry into the methods doc: OpenAlex holds only the support years it has indexed (Galaxy: -10 to -13, not -01 to -09). The collapsed award therefore carries the years we captured, not the project's full value.
NIH awards remain understated relative to HHS. Collapsing narrows the grain mismatch; it does not close it.

In [ ]:
NIH_ROR = "https://ror.org/01cwqze88"

_pre_rows, _pre_usd = len(df), df["AMOUNT_USD"].sum()

df["nih_core_project"] = None
_is_nih = df["FUNDER_ROR"].eq(NIH_ROR)
df.loc[_is_nih, "nih_core_project"] = df.loc[_is_nih, "GRANT_ID"].map(nih_core_project)

_collapsible = df["nih_core_project"].notna()
print(f"NIH rows with a support-year identifier: {_collapsible.sum()}")
print(f"Distinct core projects: {df.loc[_collapsible, 'nih_core_project'].nunique()}")

if _collapsible.any():
    # Collapse within (core project, OI) so multi-recipient structure survives.
    grp = ["nih_core_project", "OI"]

    agg = {
        "AMOUNT": "sum",
        "AMOUNT_USD": "sum",
        "START_DATE": "min",
        "END_DATE": "max",
        "GRANT_ID": lambda s: sorted(s)[0],          # earliest support year
        "record_uid": "first",
        "award_cluster_id": "first",
    }
    # Everything else: take the first value. Verified constant within group.
    passthrough = [
        c for c in df.columns
        if c not in agg and c not in grp and c != "nih_source_grant_ids"
    ]
    for col in passthrough:
        agg[col] = "first"

    nih = df[_collapsible].copy()
    nih["nih_source_grant_ids"] = nih["GRANT_ID"]

    collapsed = nih.groupby(grp, as_index=False).agg(agg)
    # Provenance: every support-year ID that fed the collapsed row.
    ids = (
        nih.groupby(grp)["nih_source_grant_ids"]
        .apply(lambda s: "; ".join(sorted(map(str, s))))
        .reset_index()
    )
    collapsed = collapsed.merge(ids, on=grp, how="left")
    collapsed["nih_increments_collapsed"] = (
        collapsed["nih_source_grant_ids"].str.count(";") + 1
    )

    df = pd.concat([df[~_collapsible], collapsed], ignore_index=True)

df["nih_increments_collapsed"] = df.get(
    "nih_increments_collapsed", pd.Series(index=df.index, dtype="float")
).fillna(1).astype(int)

print(f"\nRows: {_pre_rows} -> {len(df)}  ({_pre_rows - len(df)} collapsed)")
print(f"USD:  ${_pre_usd:,.0f} -> ${df['AMOUNT_USD'].sum():,.0f}")
assert abs(df["AMOUNT_USD"].sum() - _pre_usd) < 1, "Collapse changed total USD"

NIH rows with a support-year identifier: 17
Distinct core projects: 8

Rows: 1175 -> 1166  (9 collapsed)
USD:  $715,486,483 -> $715,486,483


## Funder canonicalisation, region, US federal flag

Two funder-name problems, one fix.
1. OpenAIRE and OpenAlex wrap funder names in square brackets, so "[National Science Foundation]" and "National Science Foundation" are distinct string keys. Grouping on FUNDER splits NSF's dollars in two.
2. Some funders have genuinely different strings for the same body ("Agence Nationale de la Recherche" / "French National Research Agency (ANR)"). Bracket-stripping does not fix these.

Both resolve by joining on FUNDER_ROR and taking canonical_name from the
crosswalk. Group on the ROR, label with the canonical name. Never group on
the raw string.

* region ∈ {Europe, US, Rest of world}, where Europe means EU27 + UK + Switzedland, Norway, and other non-EU European states

In [ ]:
STUB_ROR = "NO_ROR:sergey_brin_family_foundation"
crosswalk = pd.read_csv(ROR_EXT_FILE)

# The FX cell writes '' (not null) when a funder name doesn't resolve to a ROR.
# Fold blanks into null so both reach the stub key below.
df["FUNDER_ROR"] = df["FUNDER_ROR"].replace("", pd.NA)
df["FUNDER_ROR"] = df["FUNDER_ROR"].fillna(STUB_ROR)

_before = len(df)
df = df.merge(
    crosswalk[
        ["funder_ror", "canonical_name", "country_code", "region", "is_us_federal"]
    ].rename(
        columns={
            "funder_ror": "FUNDER_ROR",
            "canonical_name": "FUNDER_CANONICAL",
            "country_code": "FUNDER_COUNTRY",
            "region": "FUNDER_REGION",
            "is_us_federal": "IS_US_FEDERAL",
        }
    ),
    on="FUNDER_ROR",
    how="left",
)
assert len(df) == _before, "Crosswalk join changed row count — duplicate funder_ror?"

_unmatched = df["FUNDER_CANONICAL"].isna()
if _unmatched.any():
    raise ValueError(
        f"{_unmatched.sum()} rows have no crosswalk entry:\n"
        f"{df.loc[_unmatched, 'FUNDER_ROR'].value_counts().to_string()}"
    )

df["IS_US_FEDERAL"] = df["IS_US_FEDERAL"].fillna(False).astype(bool)

print("Funder strings collapsed: "
      f"{df['FUNDER'].nunique()} raw -> {df['FUNDER_CANONICAL'].nunique()} canonical")
print(f"\nAwards by region:\n{df['FUNDER_REGION'].value_counts().to_string()}")
print(f"\nUSD by region:\n{df.groupby('FUNDER_REGION')['AMOUNT_USD'].sum().map('${:,.0f}'.format).to_string()}")
print(f"\nUS federal: {df['IS_US_FEDERAL'].sum()} awards, "
      f"${df.loc[df.IS_US_FEDERAL, 'AMOUNT_USD'].sum():,.0f}")

# Sanity: the 8 federal funders should be exactly the ones we expect.
print("\nFederal funders present:")
print(df.loc[df.IS_US_FEDERAL, "FUNDER_CANONICAL"].value_counts().to_string())

_n_fed = int(df["IS_US_FEDERAL"].sum())
print(f"US federal awards: {_n_fed}")
assert 0 < df.loc[df.IS_US_FEDERAL, 'FUNDER_CANONICAL'].nunique() <= 8, \
    "Federal funder count off — check IS_US_FEDERAL parsing"


Funder strings collapsed: 51 raw -> 47 canonical

Awards by region:
FUNDER_REGION
US               741
Europe           350
Rest of world     75

USD by region:
FUNDER_REGION
Europe           $185,059,118
Rest of world      $4,531,125
US               $525,896,240

US federal: 641 awards, $470,043,172

Federal funders present:
FUNDER_CANONICAL
U.S. National Science Foundation                         418
Institute of Museum and Library Services                  86
United States Department of Health and Human Services     60
National Endowment for the Humanities                     59
National Aeronautics and Space Administration              8
National Institutes of Health                              8
United States Department of Defense                        1
United States Department of Energy                         1
US federal awards: 641


## Report + write

In [ ]:
print(f'Rows: {len(df):,} | multi-recipient rows: {int(df.MULTI_RECIPIENT.sum()):,}')
print(f'  AMOUNT_USD populated: {int(df.AMOUNT_USD.notna().sum()):,} (blank/counts-only: {int(df.AMOUNT_USD.isna().sum()):,})')
print(f'  FUNDER_ROR matched: {int((df.FUNDER_ROR.astype(str).str.len()>0).sum()):,} (funders only)')
print(f'  SOLUTION_CATEGORY populated: {int((df.SOLUTION_CATEGORY.astype(str).str.len()>0).sum()):,} / {len(df):,}')
eco=df.drop_duplicates("award_cluster_id")["AMOUNT_USD"].sum()
print(f'  ecosystem total (unique awards): {eco:,.0f} | sum DASH_AMT (per-OI safe): {df.DASH_AMT.sum():,.0f}')
df.to_csv(OUT_ENRICH,index=False); print('Wrote',OUT_ENRICH)

# --- award x category 'reach' export (overlaps by design; NEVER sum across categories for a total) ---
aw=df.drop_duplicates('award_cluster_id').copy()
keep=[c for c in ['award_cluster_id',COL_FUNDER,COL_YEAR,'AMOUNT_USD'] if c in aw.columns]
aw=aw[keep+['AWARD_SOLUTION_CATEGORY']]
aw['solution_category']=aw['AWARD_SOLUTION_CATEGORY'].fillna('').str.split('; ')
cat_long=aw.explode('solution_category')
cat_long=cat_long[cat_long['solution_category'].astype(str).str.len()>0].drop(columns='AWARD_SOLUTION_CATEGORY')
cat_long.to_csv(OUT_CAT,index=False)
Path(str(OUT_CAT)+'.README.txt').write_text(
 'grant_solution_categories.csv -- funding REACHING each solution category.\n'
 'The full award AMOUNT_USD is counted ONCE in EVERY category the award touches.\n'
 'Rows OVERLAP by design: summing AMOUNT_USD across categories exceeds the ecosystem total.\n'
 'These figures measure REACH, not a division of the total. Use independent bars, never a pie.\n'
 'For once-per-award totals and $/funder use enriched.csv (sum AMOUNT_USD over unique award_cluster_id).\n')
catsum=cat_long.groupby('solution_category')['AMOUNT_USD'].sum().sort_values(ascending=False)
print(f'Wrote {OUT_CAT}  ({len(cat_long):,} award x category rows; clean CSV + .README.txt caveat)')
print(f'  reach sum across categories = {catsum.sum():,.0f}  vs ecosystem total {eco:,.0f}  (expected to EXCEED it)')

Rows: 1,166 | multi-recipient rows: 122
  AMOUNT_USD populated: 1,057 (blank/counts-only: 109)
  FUNDER_ROR matched: 1,166 (funders only)
  SOLUTION_CATEGORY populated: 1,166 / 1,166
  ecosystem total (unique awards): 677,349,039 | sum DASH_AMT (per-OI safe): 623,784,079
Wrote /content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/enriched.csv
Wrote /content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/grant_solution_categories.csv  (1,862 award x category rows; clean CSV + .README.txt caveat)
  reach sum across categories = 1,164,369,892  vs ecosystem total 677,349,039  (expected to EXCEED it)


## Diagnostics: top 20 by amount

In [ ]:
mm = pd.read_csv(OUT_ENRICH)
mm.drop_duplicates('award_cluster_id').nlargest(20,'AMOUNT_USD')[['OI','FUNDER','AMOUNT_USD','CURRENCY','GRANT_YEAR']]

,OI,FUNDER,AMOUNT_USD,CURRENCY,GRANT_YEAR
487,research resource identification initiative,Department of Health and Human Services,2.062476e+07,USD,2012.0
1158,icpsr,[National Institutes of Health],1.674516e+07,USD,2002.0
1056,dspace,National Science Foundation,1.500000e+07,USD,2014.0
1024,crossref,National Aeronautics and Space Administration,1.367258e+07,USD,2018.0
752,irods,National Science Foundation,8.300992e+06,USD,2011.0
356,osf (open science framework),Department of Health and Human Services,7.997459e+06,USD,1992.0
312,openalex,Arcadia Fund,7.500000e+06,USD,2024.0
662,icpsr,Department of Health and Human Services,7.489653e+06,USD,2020.0
817,synapse,Department of Health and Human Services,7.311900e+06,USD,2015.0
1165,galaxy,[National Institutes of Health],7.134957e+06,USD,2012.0


## Diagnostic: European Commmission grants >$2M

In [ ]:
aw = mm.drop_duplicates('award_cluster_id')
ec = aw[aw['FUNDER'].str.contains('European Commission', case=False, na=False)
        & (pd.to_numeric(aw['AMOUNT_USD'], errors='coerce') > 700000)]
print(f"EC awards > $5M: {len(ec)} | total ${pd.to_numeric(ec['AMOUNT_USD']).sum():,.0f}")
ec[['OI','FUNDER','AMOUNT_USD','GRANT_YEAR']].sort_values('AMOUNT_USD', ascending=False)

EC awards > $5M: 12 | total $14,029,409


,OI,FUNDER,AMOUNT_USD,GRANT_YEAR
979,openaire graph,[European Commission],2.721713e+06,2021.0
34,d4science infrastructure,[European Commission],1.512461e+06,2011.0
833,d4science infrastructure,[European Commission],1.343047e+06,2009.0
525,pundit,[European Commission],1.188330e+06,2014.0
240,europe pmc,[European Commission],1.174600e+06,2021.0
188,datacite,[European Commission],1.042909e+06,2017.0
321,europe pmc,[European Commission],9.717200e+05,2016.0
907,zenodo,[European Commission],9.373010e+05,2025.0
295,d4science infrastructure,[European Commission],8.600744e+05,2008.0
607,openaire graph,[European Commission],7.876694e+05,2024.0


In [ ]:
tot = pd.to_numeric(aw['AMOUNT_USD'], errors='coerce').sum()
print(f"these {len(ec)} awards = ${pd.to_numeric(ec['AMOUNT_USD']).sum():,.0f}  ({pd.to_numeric(ec['AMOUNT_USD']).sum()/tot:.1%} of ecosystem total)")

these 28 awards = $228,614,643  (23.9% of ecosystem total)


## Write manual review sheet

Manually review top 20 by amount and EC grants >$2M as they have huge influence on the analysis, and EC grants >$2M tend to be consortial in nature.

In [ ]:
mm = pd.read_csv(OUT_ENRICH)
aw = mm.drop_duplicates('award_cluster_id').copy()
aw['AMOUNT_USD'] = pd.to_numeric(aw['AMOUNT_USD'], errors='coerce')

top20 = set(aw.nlargest(20, 'AMOUNT_USD')['award_cluster_id'])
ec2m  = set(aw[aw['FUNDER'].str.contains('European Commission', case=False, na=False)
              & (aw['AMOUNT_USD'] > 2_000_000)]['award_cluster_id'])

flagged = aw[aw['award_cluster_id'].isin(top20 | ec2m)].copy()
flagged['flag_reason'] = flagged['award_cluster_id'].map(
    lambda c: '+'.join(f for f,s in [('top20',top20),('EC>2M',ec2m)] if c in s))
flagged['correct_supercat'] = flagged['pred_supercat']     # pre-fill from current label
for col in ['is_consortial_yn','oi_share_amount','source_url','note']:
    flagged[col] = ''

flagged = flagged.sort_values('AMOUNT_USD', ascending=False)
cols = ['award_cluster_id','flag_reason','OI','FUNDER','AMOUNT_USD','GRANT_YEAR',
        'pred_supercat','pred_cat','correct_supercat','is_consortial_yn',
        'oi_share_amount','source_url','note','TITLE','DESCRIPTION']
out = flagged[cols]
out.to_csv(BASE_DIR/'consortium_review_queue.csv', index=False)
print(f"{len(out)} awards -> consortium_review_queue.csv "
      f"(${out['AMOUNT_USD'].sum():,.0f}, {out['AMOUNT_USD'].sum()/aw['AMOUNT_USD'].sum():.1%} of total)")
print(f"overlap: top20-only {len(top20-ec2m)}, EC>2M-only {len(ec2m-top20)}, both {len(top20&ec2m)}")

38 awards -> consortium_review_queue.csv ($357,315,393, 37.3% of total)
overlap: top20-only 10, EC>2M-only 18, both 10


In [ ]:
# Override-apply manual annotations

import numpy as np

# ── CONFIG ─────────────────────────────────────────────
REVIEW_PATH      = BASE_DIR/'consortium_review_queue.csv'                 # completed sheet
CATEGORISED_PATH = BASE_DIR/'deduplicated_clean_categorised.csv'          # pre-enrich input
CORRECTED_PATH   = BASE_DIR/'deduplicated_clean_categorised_corrected.csv'# enrich reads THIS after

# What to do when a grant is consortial but the OI's share can't be recovered:
#   'counts_only' -> drop the inflated $ (blank AMOUNT), keep the grant in counts   [safer default]
#   'keep_full'   -> leave the full amount (knowingly over-attributed)              [team's call]
UNRECOVERABLE_SHARE = 'counts_only'
SUPERCATS = ['DIRECT','ADJACENT','ADOPT','UNKNOWN']
DIRECT_ACT = ['RESEARCH & DEVELOPMENT','OPERATIONS','OTHER']
# ───────────────────────────────────────────────────────

review = pd.read_csv(REVIEW_PATH, dtype=str)
cat    = pd.read_csv(CATEGORISED_PATH, dtype=str)

# only rows actually reviewed (is_consortial_yn filled)
rv = review[review['is_consortial_yn'].astype(str).str.strip().str.lower().isin(['y','yes','n','no'])].copy()
print(f"reviewed rows: {len(rv)} / {len(review)}")

supercat_ovr, amount_ovr, countsonly = {}, {}, set()
for _, r in rv.iterrows():
    cid  = str(r['award_cluster_id'])
    cons = str(r['is_consortial_yn']).strip().lower() in ('y','yes')
    cs   = str(r.get('correct_supercat','') or '').strip().upper()
    if cs in SUPERCATS:
        supercat_ovr[cid] = cs
    share = pd.to_numeric(r.get('oi_share_amount'), errors='coerce')
    if cons:
        if pd.notna(share): amount_ovr[cid] = share        # in the grant's ORIGINAL currency
        else:               countsonly.add(cid)

cat['award_cluster_id'] = cat['award_cluster_id'].astype(str)
cat['AMOUNT_ORIG'] = cat['AMOUNT']          # preserve original, always
cat['override_note'] = ''

# 1) reclassification — fix supercat AND keep leaf-cat consistent
m = cat['award_cluster_id'].isin(supercat_ovr)
newsup = cat.loc[m,'award_cluster_id'].map(supercat_ovr)
cat.loc[m,'pred_supercat'] = newsup
#    if moved OFF direct, leaf must equal the supercat; if onto direct, leave existing activity
off_direct = m & ~newsup.isin(['DIRECT']).reindex(cat.index, fill_value=False)
cat.loc[m & (newsup!='DIRECT').reindex(cat.index, fill_value=False), 'pred_cat'] = \
    cat.loc[m & (newsup!='DIRECT').reindex(cat.index, fill_value=False),'pred_supercat']
cat.loc[m,'override_note'] += 'supercat_corrected; '

# 2) dollar re-attribution — inject OI share into AMOUNT, keep CURRENCY so enrich converts consistently
m = cat['award_cluster_id'].isin(amount_ovr)
cat.loc[m,'AMOUNT'] = cat.loc[m,'award_cluster_id'].map(amount_ovr).astype(str)
cat.loc[m,'override_note'] += 'amount_reattributed; '

# 3) consortial but share unrecoverable
if UNRECOVERABLE_SHARE == 'counts_only':
    m = cat['award_cluster_id'].isin(countsonly)
    cat.loc[m,'AMOUNT'] = ''    # blank -> enrich yields blank AMOUNT_USD -> counts-only
    cat.loc[m,'override_note'] += 'consortial_share_unknown->counts_only; '

cat.to_csv(CORRECTED_PATH, index=False)
print(f"reclassified: {len(supercat_ovr)} | $ re-attributed: {len(amount_ovr)} | "
      f"counts-only (share unknown): {len(countsonly)}")
print(f"wrote {CORRECTED_PATH}")
print("→ set enrich's IN_DEDUP to this file and RE-RUN enrich, then analysis reads the corrected totals.")

reviewed rows: 38 / 38
reclassified: 38 | $ re-attributed: 17 | counts-only (share unknown): 16
wrote /content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/deduplicated_clean_categorised_corrected.csv
→ set enrich's IN_DEDUP to this file and RE-RUN enrich, then analysis reads the corrected totals.


In [ ]:
import pandas as pd
CIDS = ['2666', '730']   # a few that look dropped

rev = pd.read_csv(REVIEW_PATH, dtype=str)
enr = pd.read_csv(OUT_ENRICH)

print("=== review sheet ===")
print(rev[rev.award_cluster_id.astype(str).isin(CIDS)]
      [['award_cluster_id','is_consortial_yn','oi_share_amount_USD','correct_supercat']].to_string())

print("\n=== enriched output ===")
cols = ['award_cluster_id','OI','AMOUNT','AMOUNT_ORIG','CURRENCY','AMOUNT_USD',
        'MULTI_RECIPIENT','DASH_AMT','override_note']
print(enr[enr.award_cluster_id.astype(str).isin(CIDS)]
      [[c for c in cols if c in enr.columns]].to_string())

=== review sheet ===
   award_cluster_id is_consortial_yn oi_share_amount_USD correct_supercat
20             2666                y           832804.97           DIRECT
22              730                y           759270.96           DIRECT

=== enriched output ===
      award_cluster_id              OI  AMOUNT  AMOUNT_ORIG CURRENCY  AMOUNT_USD  MULTI_RECIPIENT  DASH_AMT                                                override_note
607               2666  openaire graph     NaN    9999980.0      EUR         NaN            False       NaN  supercat_corrected; consortial_share_unknown->counts_only; 
1049               730  openaire graph     NaN    7274150.0      EUR         NaN            False       NaN  supercat_corrected; consortial_share_unknown->counts_only; 
